# Quantum Entropy Harvester

Harvests real quantum entropy from IBM Quantum hardware (ibm_fez, ibm_marrakesh — 156-qubit Heron r2 processors) into Zipminator's ever-growing entropy pool.

**Two modes:**
- **One-shot**: Run once, harvest entropy, append to pool
- **Scheduled**: Loop every N minutes, harvesting continuously

**Where to run:**
- **Locally**: Uses IBM Quantum credentials from `~/.qiskit/qiskit-ibm.json`
- **qBraid Lab**: Upload this notebook, credentials are automatic
- **Any Jupyter**: Just needs `qiskit-ibm-runtime` installed

**Budget guard**: 8 min of 10 min/month free QPU time (configurable)

In [ ]:
# Configuration
POOL_PATH = "../../quantum_entropy/quantum_entropy_pool.bin"  # Relative to notebook
LOG_PATH = "../../quantum_entropy/harvest_log.jsonl"
BUDGET_PATH = "../../quantum_entropy/ibm_budget.json"

NUM_QUBITS = 16          # Qubits per circuit (16 = 2 bytes per shot)
SHOTS = 1024             # Shots per circuit (1024 shots = 2048 bytes)
MAX_QPU_SECONDS = 480    # 8 min of 10 min free tier
QPU_MS_PER_SHOT = 1.0    # Conservative estimate

# For scheduled mode:
HARVEST_INTERVAL_MINUTES = 60  # Harvest every hour
MAX_HARVESTS = 24              # Stop after 24 harvests (24 hours)

# Preferred backends (in order)
PREFERRED_BACKENDS = ["ibm_fez", "ibm_marrakesh", "ibm_kingston", "ibm_brisbane"]

In [ ]:
import json
import hashlib
import os
import time
from datetime import datetime, timezone
from pathlib import Path

# Resolve paths (works both locally and in qBraid Lab)
pool_path = Path(POOL_PATH).resolve()
log_path = Path(LOG_PATH).resolve()
budget_path = Path(BUDGET_PATH).resolve()

# If paths don't exist (running in qBraid Lab), use local directory
if not pool_path.parent.exists():
    pool_path = Path("quantum_entropy_pool.bin")
    log_path = Path("harvest_log.jsonl")
    budget_path = Path("ibm_budget.json")
    print(f"Remote mode: saving to current directory")
else:
    print(f"Local mode: pool at {pool_path}")

print(f"Pool exists: {pool_path.exists()}, size: {pool_path.stat().st_size if pool_path.exists() else 0} bytes")

In [ ]:
# --- Budget Guard ---

def current_month():
    return datetime.now(timezone.utc).strftime("%Y-%m")

def load_budget():
    month = current_month()
    if budget_path.exists():
        try:
            data = json.loads(budget_path.read_text())
            if data.get("month") == month:
                return data
        except (json.JSONDecodeError, KeyError):
            pass
    return {"month": month, "cumulative_seconds": 0.0, "jobs": []}

def save_budget(data):
    budget_path.write_text(json.dumps(data, indent=2))

def check_budget(estimated_seconds):
    budget = load_budget()
    used = budget["cumulative_seconds"]
    projected = used + estimated_seconds
    if projected > MAX_QPU_SECONDS:
        print(f"BUDGET EXCEEDED: {used:.1f}s used + {estimated_seconds:.1f}s = {projected:.1f}s > {MAX_QPU_SECONDS}s limit")
        return False
    pct = used / MAX_QPU_SECONDS * 100
    print(f"Budget: {used:.1f}s / {MAX_QPU_SECONDS}s ({pct:.0f}%) — {estimated_seconds:.1f}s estimated for this job")
    return True

def record_job(job_id, seconds):
    budget = load_budget()
    budget["cumulative_seconds"] += seconds
    budget["jobs"].append({"job_id": job_id, "seconds": seconds, "timestamp": datetime.now(timezone.utc).isoformat()})
    save_budget(budget)

print(f"Budget this month: {load_budget()['cumulative_seconds']:.1f}s / {MAX_QPU_SECONDS}s")

In [ ]:
# --- Connect to Quantum Hardware ---
# Tries: 1) Rigetti via qBraid (free)  2) IBM via qBraid  3) IBM direct

backend = None
service = None
use_qbraid_rigetti = False

# --- Option 1: Rigetti via qBraid (FREE, no QPU limit) ---
try:
    from qbraid.runtime import QbraidProvider
    provider = QbraidProvider()
    devices = provider.get_devices()
    print("qBraid devices found:")
    for d in devices:
        status = getattr(d, 'status', lambda: 'unknown')
        print(f"  {d.id}: {getattr(d, 'name', d.id)} ({getattr(d, 'num_qubits', '?')}q)")
    
    # Look for Rigetti
    rigetti = [d for d in devices if 'rigetti' in str(d.id).lower() or 'ankaa' in str(d.id).lower()]
    if rigetti:
        backend = rigetti[0]
        use_qbraid_rigetti = True
        print(f"\nSelected: {backend.id} (Rigetti via qBraid - FREE)")
    else:
        print("\nNo Rigetti device found via qBraid")
except Exception as e:
    print(f"qBraid not available: {e}")

# --- Option 2: IBM via qiskit-ibm-runtime ---
if backend is None:
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService
        for channel in ['ibm_quantum', 'ibm_cloud', None]:
            try:
                service = QiskitRuntimeService(channel=channel) if channel else QiskitRuntimeService()
                print(f"\nIBM Quantum connected via {channel or 'default'}")
                break
            except Exception:
                continue
        
        if service:
            backends = service.backends(simulator=False)
            operational = [b for b in backends if b.num_qubits >= NUM_QUBITS]
            print(f"IBM backends ({len(backends)} total, {len(operational)} with >={NUM_QUBITS}q):")
            for b in sorted(operational, key=lambda x: x.num_qubits, reverse=True)[:8]:
                print(f"  {b.name}: {b.num_qubits}q")
            if operational:
                backend = min(operational, key=lambda b: b.status().pending_jobs)
                print(f"\nSelected: {backend.name} ({backend.num_qubits}q)")
    except Exception as e:
        print(f"IBM Quantum not available: {e}")

if backend:
    print(f"\n>>> Ready to harvest from: {backend.id if use_qbraid_rigetti else backend.name}")
else:
    print("\nNo quantum backend available!")

In [ ]:
# --- Build Hadamard Circuit ---

from qiskit import QuantumCircuit

qc = QuantumCircuit(NUM_QUBITS, NUM_QUBITS)
for i in range(NUM_QUBITS):
    qc.h(i)
for i in range(NUM_QUBITS):
    qc.measure(i, i)

print(f"Circuit: H^{NUM_QUBITS} + measure")
print(f"Expected entropy: {SHOTS * NUM_QUBITS // 8} bytes ({SHOTS} shots x {NUM_QUBITS} bits / 8)")
print(qc.draw(output='text'))

In [ ]:
# --- Harvest Function (supports both Rigetti/qBraid and IBM) ---

def harvest_once_rigetti(device, num_qubits=NUM_QUBITS, shots=SHOTS):
    """Harvest via qBraid Rigetti backend."""
    from qiskit import QuantumCircuit, qasm2
    from qbraid.runtime.native.device import Program as NativeProgram

    estimated_seconds = shots * QPU_MS_PER_SHOT / 1000
    if not check_budget(estimated_seconds):
        return 0, None

    # Build circuit
    qc = QuantumCircuit(num_qubits, num_qubits)
    for i in range(num_qubits):
        qc.h(i)
    for i in range(num_qubits):
        qc.measure(i, i)

    # Convert to QASM2 string, then wrap in the native Program pydantic model
    qasm_str = qasm2.dumps(qc)
    program = NativeProgram(format="qasm2", data=qasm_str)
    print(f"Built NativeProgram(format='qasm2') for {device.id}")

    print(f"Submitting H^{num_qubits} to {device.id} ({shots} shots)...")
    t0 = time.time()

    # Submit directly via device client to bypass the broken device.run() path
    from qbraid.runtime.native.device import JobRequest
    job_request = JobRequest(
        deviceQrn=device.id,
        program=program,
        shots=shots,
    )
    job_data = device.client.create_job(job_request)
    from qbraid.runtime.native.job import QbraidJob
    job = QbraidJob(job_id=job_data.jobQrn, device=device, client=device.client)
    job_id = job.id
    print(f"Job ID: {job_id}")

    result = job.result()
    elapsed = time.time() - t0
    print(f"Completed in {elapsed:.1f}s")

    # Extract counts
    counts = {}
    for method in ['get_counts', 'measurement_counts']:
        fn = getattr(result, method, None)
        if fn and callable(fn):
            try:
                counts = fn()
                if counts:
                    break
            except Exception:
                continue
    if not counts and hasattr(result, 'data'):
        try:
            counts = result.data.get_counts()
        except Exception:
            pass

    if not counts:
        print(f"Could not extract counts. Result type: {type(result)}")
        print(f"Result attrs: {[x for x in dir(result) if not x.startswith('_')]}")
        return 0, None

    # Convert to entropy bytes
    entropy_bytes = bytearray()
    for bitstring, count in counts.items():
        bits = str(bitstring).replace(' ', '')
        n_bytes = max(1, len(bits) // 8)
        byte_val = int(bits, 2).to_bytes(n_bytes, byteorder='big')
        for _ in range(count):
            entropy_bytes.extend(byte_val)

    entropy = bytes(entropy_bytes)
    sha = hashlib.sha256(entropy).hexdigest()

    # Append to pool
    pool_path.parent.mkdir(parents=True, exist_ok=True)
    pool_before = pool_path.stat().st_size if pool_path.exists() else 0
    with open(pool_path, 'ab') as f:
        f.write(entropy)
    pool_after = pool_path.stat().st_size

    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "source": f"rigetti/{device.id}",
        "job_id": str(job_id),
        "qubits": num_qubits, "shots": shots,
        "circuit": f"H^{num_qubits}",
        "entropy_bytes": len(entropy), "sha256": sha,
        "elapsed_seconds": round(elapsed, 1),
        "pool_before": pool_before, "pool_after": pool_after,
        "certification": "born_rule"
    }
    with open(log_path, 'a') as f:
        f.write(json.dumps(record) + '\n')

    record_job(str(job_id), estimated_seconds)
    print(f"\nHarvested {len(entropy)} bytes from {device.id}")
    print(f"SHA-256: {sha[:16]}...")
    print(f"Pool: {pool_before} -> {pool_after} bytes")
    print(f"Unique bitstrings: {len(counts)}")
    return len(entropy), str(job_id)


def harvest_once_ibm(backend, circuit, shots=SHOTS):
    """Harvest via IBM Quantum (qiskit-ibm-runtime)."""
    from qiskit_ibm_runtime import SamplerV2 as Sampler
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    estimated_seconds = shots * QPU_MS_PER_SHOT / 1000
    if not check_budget(estimated_seconds):
        return 0, None

    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    transpiled = pm.run(circuit)
    sampler = Sampler(mode=backend)
    print(f"Submitting to {backend.name}...")
    t0 = time.time()
    job = sampler.run([transpiled], shots=shots)
    print(f"Job ID: {job.job_id()}")
    result = job.result()
    elapsed = time.time() - t0
    print(f"Completed in {elapsed:.1f}s")

    counts = result[0].data.meas.get_counts()
    entropy_bytes = bytearray()
    for bitstring, count in counts.items():
        bits = bitstring.replace(' ', '')
        byte_val = int(bits, 2).to_bytes(NUM_QUBITS // 8, byteorder='big')
        for _ in range(count):
            entropy_bytes.extend(byte_val)

    entropy = bytes(entropy_bytes)
    sha = hashlib.sha256(entropy).hexdigest()

    pool_path.parent.mkdir(parents=True, exist_ok=True)
    pool_before = pool_path.stat().st_size if pool_path.exists() else 0
    with open(pool_path, 'ab') as f:
        f.write(entropy)
    pool_after = pool_path.stat().st_size

    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "source": f"ibm_quantum/{backend.name}",
        "job_id": job.job_id(),
        "qubits": NUM_QUBITS, "shots": shots,
        "circuit": f"H^{NUM_QUBITS}",
        "entropy_bytes": len(entropy), "sha256": sha,
        "elapsed_seconds": round(elapsed, 1),
        "pool_before": pool_before, "pool_after": pool_after,
        "certification": "born_rule"
    }
    with open(log_path, 'a') as f:
        f.write(json.dumps(record) + '\n')

    record_job(job.job_id(), estimated_seconds)
    print(f"\nHarvested {len(entropy)} bytes from {backend.name}")
    print(f"Pool: {pool_before} -> {pool_after} bytes")
    return len(entropy), job.job_id()


def harvest_once(backend, circuit=None, shots=SHOTS):
    """Auto-dispatch to Rigetti or IBM."""
    if use_qbraid_rigetti:
        return harvest_once_rigetti(backend, shots=shots)
    else:
        return harvest_once_ibm(backend, circuit, shots=shots)

print("Harvest functions ready.")

## One-Shot Harvest
Run the cell below to harvest once.

In [ ]:
# --- ONE-SHOT HARVEST ---
bytes_harvested, job_id = harvest_once(backend, qc)
print(f"\nDone! {bytes_harvested} bytes harvested, job {job_id}")

## Scheduled Harvest (Loop)
Run the cell below to harvest continuously every `HARVEST_INTERVAL_MINUTES` minutes.
It will stop after `MAX_HARVESTS` cycles or when the budget is exhausted.

In [ ]:
# --- SCHEDULED HARVEST ---
import IPython.display as display

total_bytes = 0
total_jobs = 0

for cycle in range(1, MAX_HARVESTS + 1):
    display.clear_output(wait=True)
    print(f"=== Harvest cycle {cycle}/{MAX_HARVESTS} ===")
    print(f"Total harvested so far: {total_bytes:,} bytes from {total_jobs} jobs")
    print(f"Time: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print()
    
    try:
        harvested, jid = harvest_once(backend, qc)
        if harvested == 0:
            print("Budget exhausted. Stopping.")
            break
        total_bytes += harvested
        total_jobs += 1
    except Exception as e:
        print(f"Error: {e}")
        print("Will retry next cycle.")
    
    if cycle < MAX_HARVESTS:
        print(f"\nSleeping {HARVEST_INTERVAL_MINUTES} minutes until next harvest...")
        time.sleep(HARVEST_INTERVAL_MINUTES * 60)

print(f"\n=== DONE ===")
print(f"Total: {total_bytes:,} bytes from {total_jobs} jobs")
print(f"Pool size: {pool_path.stat().st_size:,} bytes")

## Download Pool (for qBraid Lab / remote use)
If running remotely, download the pool file to transfer to your local machine.

In [ ]:
# Show pool stats
if pool_path.exists():
    size = pool_path.stat().st_size
    print(f"Pool: {pool_path}")
    print(f"Size: {size:,} bytes ({size/1024/1024:.2f} MB)")
    
    # SHA-256 for verification
    sha = hashlib.sha256(pool_path.read_bytes()).hexdigest()
    print(f"SHA-256: {sha}")
    
    # In Jupyter, create download link
    try:
        from IPython.display import FileLink
        display.display(FileLink(str(pool_path), result_html_prefix='Download: '))
    except Exception:
        print(f"\nTo download: copy {pool_path} to your local machine")
else:
    print("No pool file found.")

## Append Remote Pool to Local Pool
If you downloaded a pool file from qBraid Lab, run this locally to merge it.

In [ ]:
# --- MERGE REMOTE POOL INTO LOCAL ---
# Set this to the path of the downloaded file
REMOTE_POOL = "quantum_entropy_pool.bin"  # Change to your download path

local_pool = Path("../../quantum_entropy/quantum_entropy_pool.bin").resolve()
remote_pool = Path(REMOTE_POOL)

if remote_pool.exists() and local_pool.exists():
    remote_bytes = remote_pool.read_bytes()
    local_before = local_pool.stat().st_size
    with open(local_pool, 'ab') as f:
        f.write(remote_bytes)
    local_after = local_pool.stat().st_size
    print(f"Appended {len(remote_bytes):,} bytes")
    print(f"Local pool: {local_before:,} -> {local_after:,} bytes")
elif not remote_pool.exists():
    print(f"Remote pool not found at {REMOTE_POOL}")
elif not local_pool.exists():
    print(f"Local pool not found. Copying remote as new pool.")
    local_pool.parent.mkdir(parents=True, exist_ok=True)
    local_pool.write_bytes(remote_pool.read_bytes())